In [ ]:
import random
from collections import Counter

# ----------------------------
# Domain knowledge (ecology)
# ----------------------------

PLANT_SPECIES = {
    "grass": {
        "edibility": 0.8,
        "growth_rate": 0.20
    },
    "clover": {
        "edibility": 0.6,
        "growth_rate": 0.15
    },
    "shrub": {
        "edibility": 0.2,
        "growth_rate": 0.05
    }
}

ANIMAL_SPECIES = {
    "rabbit": ["grass", "clover"],
    "deer": ["grass", "clover", "shrub"]
}

LOCATIONS = ["meadow", "forest_edge", "clearing"]

TIME_STEPS = 6
NUM_TRACES = 6000


# ----------------------------
# Object definitions
# ----------------------------

class Plant:
    def __init__(self, pid):
        self.id = pid
        self.species = random.choice(list(PLANT_SPECIES.keys()))
        self.location = random.choice(LOCATIONS)
        self.alive = True
        self.eaten = False

    def step(self):
        """Plants may grow (stay alive) or die naturally"""
        if not self.alive:
            return

        # Natural death
        if random.random() < 0.03:
            self.alive = False

        # Regrowth / resilience modeled implicitly
        # (survival probability encoded in low death rate)


class Animal:
    def __init__(self, aid):
        self.id = aid
        self.species = random.choice(list(ANIMAL_SPECIES.keys()))
        self.location = random.choice(LOCATIONS)

    def move(self):
        self.location = random.choice(LOCATIONS)

    def try_eat(self, plants):
        """Attempt to eat a compatible plant in the same location"""
        for plant in plants:
            if (
                plant.alive
                and plant.location == self.location
                and plant.species in ANIMAL_SPECIES[self.species]
            ):
                p_eat = PLANT_SPECIES[plant.species]["edibility"]
                if random.random() < p_eat:
                    plant.alive = False
                    plant.eaten = True
                    return True
        return False


# ----------------------------
# World simulation
# ----------------------------

def simulate_world():
    num_animals = random.randint(1, 5)
    num_plants = random.randint(2, 6)

    animals = [Animal(i) for i in range(num_animals)]
    plants = [Plant(i) for i in range(num_plants)]

    eaten_any = False

    for _ in range(TIME_STEPS):
        for animal in animals:
            animal.move()
            if animal.try_eat(plants):
                eaten_any = True

        for plant in plants:
            plant.step()

    return {
        "plants": plants,
        "eaten_any": eaten_any
    }


# ----------------------------
# Conditioning & inference
# ----------------------------

def run_inference():
    conditioned = []

    for _ in range(NUM_TRACES):
        trace = simulate_world()
        if trace["eaten_any"]:  # observation
            conditioned.append(trace)

    return conditioned


def estimate_probabilities(traces):
    plant0_eaten = 0
    majority_survive = 0

    for trace in traces:
        plants = trace["plants"]
        alive = sum(p.alive for p in plants)

        for p in plants:
            if p.id == 0 and p.eaten:
                plant0_eaten += 1

        if alive > len(plants) / 2:
            majority_survive += 1

    total = len(traces)

    return {
        "P(plant 0 eaten | eating observed)": plant0_eaten / total,
        "P(>50% plants survive | eating observed)": majority_survive / total,
        "Conditioned traces": total
    }


 #----------------------------
# Main
# ----------------------------

if __name__ == "__main__":
    traces = run_inference()
    results = estimate_probabilities(traces)

    print("Conditioned on: at least one animal ate a plant\n")
    for k, v in results.items():
        print(f"{k}: {v:.3f}" if isinstance(v, float) else f"{k}: {v}")


# Probabilistic World Simulation: Ecosystem Model

## How Random Variables and Relations are Represented

- **Entity existence:** Number of animals (1–5) and plants (2–6) are randomly sampled at the start of each simulation.  
- **Entity attributes:** Plant species, animal species, and initial locations are randomly assigned per entity.  
- **World dynamics:** Animals move stochastically, plants may die naturally, and animals attempt to eat plants compatible with their diet.  
- **Relational structure:** Eating events depend on the interaction between animals and plants in the same location and diet compatibility.

---

## How Unknown Numbers of Objects are Handled

- The simulation uses **open-universe modeling**:  
  - Each run can have a different number of animals and plants.  
  - Each run samples species and location assignments independently.  
  - This creates a “possible world” for each trace.

---

## How Observations Affect Probabilities

- We can **condition** the simulation on the observation that **at least one animal ate a plant**.  
- Execution traces that do not satisfy this observation are discarded.  
- As a result:  
  - The probability of a specific plant being eaten increases.  
  - The probability of most plants surviving decreases.  

- This demonstrates **Bayesian reasoning under uncertainty**, using execution traces to estimate posterior probabilities.




 # Probabilistic Modeling of Student Movement on Campus

 1. Scope for a Semester-Long Project

This project aims to model and predict student movement across campus **after their scheduled activities**. Students are categorized as athletes, dancers, or regular students. The project will use **probabilistic and relational modeling** to simulate how students move between campus locations (e.g., cafeteria, library, lounges, outdoor areas) over time.  

Key objectives:
- Represent student populations with uncertain behavior.
- Capture relational dependencies (friends, teammates influence movement).
- Simulate dynamic evolution of locations over time steps.
- Condition predictions on partial observations (e.g., some students already observed at a location).

This simulation will generate **thousands of traces** to estimate probabilities and identify patterns of student movement.

---

 2. Problem Statement

Even with knowledge of student schedules, the exact **post-activity movement** is uncertain. Students make decisions influenced by personal preference, social groups, and environmental context.  

The problem is to **predict probabilistically where students are likely to go**, accounting for:
- Time-of-day effects
- Social influence from friends or teammates
- Stochastic movement choices

This allows the simulation to answer questions like:
- What is the probability that most athletes will go to the cafeteria at 1 PM?
- How does observing a few students at a location update predictions for the rest?

---

3. Relevance of AI Methodologies from the Chapter

From *Artificial Intelligence: A Modern Approach*, the following concepts apply:

- **Open-Universe Modeling**: The number and behavior of students in each location is uncertain; Monte Carlo traces sample possible worlds.
- **Dynamic Models**: Student locations evolve over discrete time steps, forming a stochastic transition process.
- **Relational Probabilistic Models**: Friend and team influences create dependencies in movement probabilities.
- **Partial Observations / Conditioning**: Observing some students updates posterior probabilities for others.
- **Monte Carlo Simulation**: Used to approximate distributions over student locations across multiple traces.
- **Bayesian Reasoning**: Inference over likely student positions given partial observations.

These methodologies allow us to **reason under uncertainty** and simulate complex, relationally-dependent behavior on campus.

---

 4. Example Python Code

The code below demonstrates **dynamic probabilistic simulation**, relational dependencies, and conditioning on observations using Monte Carlo simulation.


In [ ]:
import random
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Union

# ============================================================
# WEEK 1 — Campus Movement Simulator (clean + extensible)
# Hidden World State + Agent Internal State
# (Belief state comes in Week 2 from Monte Carlo traces)
# ============================================================

# ----------------------------
# Domain constants
# ----------------------------

STUDENT_TYPES = ["athlete", "copa", "regular"]

ATHLETE_PROGRAMS = [
    "Baseball", "TrackAndField", "Wrestling", "Soccer", "ESports",
    "Softball", "Lacrosse", "Basketball", "Volleyball", "Golf"
]

COPA_PROGRAMS = [
    "Dance", "MusicalTheater", "Acting", "Film", "Animation", "ScreenWriting"
]

LOCATIONS = [
    "LawrenceHall",
    "ThayerHall",
    "WestPenn",
    "StudentCenter",
    "Library",
    "VillagePark",
    "PlayHouse",
    "BoulevardStudios",
    "BoulevardAppartments",
    "OffCampus"
]

TIME_SLOTS = ["08:00", "10:00", "12:00", "14:00", "16:00", "18:00", "21:00"]
NUM_TIME_STEPS = len(TIME_SLOTS)

# Preferences by type and time step (simple categorical lists)
STUDENT_TYPE_PREFERENCES: Dict[str, List[List[str]]] = {
    "athlete": [
        ["VillagePark", "StudentCenter"],              # 08:00
        ["StudentCenter", "Library"],                  # 10:00
        ["StudentCenter", "BoulevardAppartments"],     # 12:00
        ["Library", "ThayerHall"],                     # 14:00
        ["VillagePark", "WestPenn"],                   # 16:00
        ["StudentCenter", "OffCampus"],                # 18:00
        LOCATIONS                                      # 21:00
    ],
    "copa": [
        ["PlayHouse", "BoulevardStudios"],             # 08:00
        ["BoulevardStudios", "PlayHouse"],             # 10:00
        ["StudentCenter", "Library"],                  # 12:00
        ["BoulevardStudios", "ThayerHall"],            # 14:00
        ["PlayHouse", "StudentCenter"],                # 16:00
        ["StudentCenter", "OffCampus"],                # 18:00
        LOCATIONS                                      # 21:00
    ],
    "regular": [LOCATIONS] * NUM_TIME_STEPS
}

# Program-level nudges (adds a small extra chance to go to these places)
PROGRAM_BOOST: Dict[str, List[str]] = {
    # athlete examples
    "TrackAndField": ["VillagePark"],
    "Soccer": ["VillagePark"],
    "Lacrosse": ["VillagePark"],
    "Baseball": ["VillagePark"],
    "Golf": ["VillagePark"],
    "Basketball": ["StudentCenter"],
    "Volleyball": ["StudentCenter"],
    "ESports": ["BoulevardAppartments", "StudentCenter"],

    # copa examples
    "Dance": ["BoulevardStudios"],
    "MusicalTheater": ["PlayHouse"],
    "Acting": ["PlayHouse"],
    "Film": ["PlayHouse", "BoulevardStudios"],
    "Animation": ["BoulevardStudios"],
    "ScreenWriting": ["Library"],
}

# --- Optional realism: "stickiness" (chance to stay in same location as last time)
STAY_PROB = 0.20  # 20% chance to stay where you were

# Safety check: each type must define exactly NUM_TIME_STEPS preference lists
for stype, prefs in STUDENT_TYPE_PREFERENCES.items():
    if len(prefs) != NUM_TIME_STEPS:
        raise ValueError(f"{stype} has {len(prefs)} time steps, expected {NUM_TIME_STEPS}")


# ----------------------------
# Time helpers (supports int index or "HH:MM")
# ----------------------------

TimeLike = Union[int, str]

def time_to_index(t: TimeLike) -> int:
    """
    Convert time into a time-step index.
    - If t is an int, it is assumed already to be the index.
    - If t is a string like "10:00", it is looked up in TIME_SLOTS.
    """
    if isinstance(t, int):
        return t
    if isinstance(t, str):
        if t not in TIME_SLOTS:
            raise ValueError(f"Unknown time slot '{t}'. Valid: {TIME_SLOTS}")
        return TIME_SLOTS.index(t)
    raise TypeError("time must be int or str like '10:00'")


# ----------------------------
# State representations
# ----------------------------


# AgentInternalState:
#   stable information about the agent (type, program, friends)
@dataclass
class AgentInternalState:
    sid: str
    stype: str
    program: str
    friends: List[str] = field(default_factory=list)

# HiddenWorldState:
#   the ground truth of the world at a time step (where everyone is)
@dataclass
class HiddenWorldState:
    t: int
    positions: Dict[str, str]  # sid -> location


# ----------------------------
# Student / Agent
# ----------------------------

@dataclass
class Student:
    internal: AgentInternalState
    location: Optional[str] = None

    def choose_location(
        self,
        prev_positions: Dict[str, str],
        time_step: int,
        friend_boost: int = 2,
        stay_prob: float = STAY_PROB,
        rng: Optional[random.Random] = None,
    ) -> str:
        """
        Week 1 movement model (simple but structured):

        Inputs:
          - prev_positions: last time-step truth (sid -> location)
          - time_step: integer index into TIME_SLOTS
          - friend_boost: repeats friends' locations to increase their probability
          - stay_prob: chance the student stays where they were last time
          - rng: Random instance for reproducibility

        Rules:
          1) With probability stay_prob, if the student had a previous location, they stay.
          2) Otherwise, we sample from a list:
             base(type,time) + friend_locs*friend_boost + program_boost_locs
        """
        rng = rng or random

        # 1) Stickiness: stay where you were (if you had a location)
        if self.location is not None and rng.random() < stay_prob:
            return self.location

        # 2) Base preferences for this type and time
        base = list(STUDENT_TYPE_PREFERENCES[self.internal.stype][time_step])

        # 3) Friend influence from previous time step
        friend_locs: List[str] = []
        for fid in self.internal.friends:
            if fid in prev_positions:
                friend_locs.append(prev_positions[fid])

        # 4) Program influence (small nudges)
        program_locs = PROGRAM_BOOST.get(self.internal.program, [])

        # Combine (friend_boost applies ONLY to friend_locs)
        possible = base + (friend_locs * friend_boost) + list(program_locs)
        return rng.choice(possible)


# ----------------------------
# Simulation
# ----------------------------

def simulate_day(
    students: List[Student],
    time_steps: int = NUM_TIME_STEPS,
    seed: Optional[int] = None
) -> List[HiddenWorldState]:
    """
    Produce ONE simulated day (one trace):
    returns a list of HiddenWorldState, one per time step.
    """
    rng = random.Random(seed)

    trace: List[HiddenWorldState] = []
    prev_positions: Dict[str, str] = {}  # none at t=0

    for t in range(time_steps):
        positions: Dict[str, str] = {}

        # shuffle order to avoid any systematic bias
        order = students[:]
        rng.shuffle(order)

        for s in order:
            loc = s.choose_location(prev_positions, time_step=t, rng=rng)
            s.location = loc
            positions[s.internal.sid] = loc

        trace.append(HiddenWorldState(t=t, positions=positions))
        prev_positions = positions

    return trace


# ----------------------------
# Conditioning (rejection sampling)
# ----------------------------

def is_consistent(trace: List[HiddenWorldState], observed: Optional[List[Dict]]) -> bool:
    """
    observed format supports BOTH:
      {"student":"a0", "time": 1,      "location":"StudentCenter"}
      {"student":"a0", "time":"10:00", "location":"StudentCenter"}

    If any observation contradicts the trace, return False.
    """
    for obs in observed or []:
        sid = obs["student"]
        t = time_to_index(obs["time"])
        loc = obs["location"]

        if t < 0 or t >= len(trace):
            return False
        if sid not in trace[t].positions:
            return False
        if trace[t].positions[sid] != loc:
            return False

    return True


def monte_carlo_simulation(
    students_template: List[Student],
    num_traces: int = 1000,
    time_steps: int = NUM_TIME_STEPS,
    observed: Optional[List[Dict]] = None,
    seed: int = 7,
) -> List[List[HiddenWorldState]]:
    """
    Week 1 Monte Carlo:
    - simulate many traces (same population each time)
    - optionally keep only traces consistent with observations (rejection sampling)

    NOTE: We COPY the students each trace so one trace doesn't leak locations into the next.
    """
    kept: List[List[HiddenWorldState]] = []
    rng = random.Random(seed)

    for _ in range(num_traces):
        # Fresh copy so locations don't carry across traces
        students = [
            Student(
                AgentInternalState(
                    sid=s.internal.sid,
                    stype=s.internal.stype,
                    program=s.internal.program,
                    friends=list(s.internal.friends),
                )
            )
            for s in students_template
        ]

        trace_seed = rng.randint(0, 10**9)
        trace = simulate_day(students, time_steps=time_steps, seed=trace_seed)

        if is_consistent(trace, observed):
            kept.append(trace)

    return kept


# ----------------------------
# Query helpers (robust: use actual internal state, not ID prefixes)
# ----------------------------

def ids_of_type(students_template: List[Student], stype: str) -> List[str]:
    return [s.internal.sid for s in students_template if s.internal.stype == stype]

def estimate_probability_majority_in_location_by_type(
    traces: List[List[HiddenWorldState]],
    students_template: List[Student],
    stype: str,
    location: str,
    time: TimeLike,
) -> float:
    """
    P( majority of students of type stype are in location at time )
    time can be an index (int) or a label like "10:00".
    """
    if not traces:
        return 0.0

    t = time_to_index(time)
    ids = ids_of_type(students_template, stype)
    if not ids:
        return 0.0

    hits = 0
    for tr in traces:
        positions = tr[t].positions
        present_ids = [sid for sid in ids if sid in positions]
        if not present_ids:
            continue

        count_loc = sum(1 for sid in present_ids if positions[sid] == location)
        if count_loc > len(present_ids) / 2:
            hits += 1

    return hits / len(traces)


# ----------------------------
# Debug printing (very helpful before Week 2)
# ----------------------------

def print_trace_by_time(trace: List[HiddenWorldState], students_template: List[Student]) -> None:
    """Print one trace grouped by time slot."""
    print("\n=== Trace by time slot ===")
    for ws in trace:
        print(f"\nTime {TIME_SLOTS[ws.t]}")
        for sid in sorted(ws.positions.keys()):
            s_obj = next(st for st in students_template if st.internal.sid == sid)
            print(f"  {sid} ({s_obj.internal.stype}, {s_obj.internal.program}): {ws.positions[sid]}")

def print_schedule_per_student(trace: List[HiddenWorldState], students_template: List[Student]) -> None:
    """Print one trace grouped by student (schedule view)."""
    print("\n=== Schedule per student ===")
    # Build sid -> list of locations over time
    sid_order = [s.internal.sid for s in students_template]
    for sid in sid_order:
        s_obj = next(st for st in students_template if st.internal.sid == sid)
        parts = []
        for ws in trace:
            loc = ws.positions.get(sid, "NA")
            parts.append(f"{TIME_SLOTS[ws.t]}->{loc}")
        print(f"{sid} ({s_obj.internal.stype}, {s_obj.internal.program}): " + " | ".join(parts))


# ----------------------------
# Demo
# ----------------------------

if __name__ == "__main__":
    # --- Build students with programs ---
    students: List[Student] = []

    # Athletes (a0, a1, a2)
    for i in range(3):
        students.append(
            Student(
                AgentInternalState(
                    sid=f"a{i}",
                    stype="athlete",
                    program=random.choice(ATHLETE_PROGRAMS),
                    friends=[f"a{(i+1)%3}"],  # simple friend ring
                )
            )
        )

    # COPA (c0, c1)
    for i in range(2):
        students.append(
            Student(
                AgentInternalState(
                    sid=f"c{i}",
                    stype="copa",
                    program=random.choice(COPA_PROGRAMS),
                    friends=[f"c{(i+1)%2}"],  # friend pair
                )
            )
        )

    # Regular (r0..r4)
    for i in range(5):
        students.append(
            Student(
                AgentInternalState(
                    sid=f"r{i}",
                    stype="regular",
                    program="None",
                    friends=[],
                )
            )
        )

    # Example: you can condition on observations using either int times or string times:
    # observed = [{"student": "a0", "time": "10:00", "location": "StudentCenter"}]
    observed = None

    # --- Run Monte Carlo simulation ---
    traces = monte_carlo_simulation(
        students_template=students,
        num_traces=500,
        time_steps=NUM_TIME_STEPS,
        observed=observed,
        seed=7
    )

    # --- Example query (clean: by type, not prefix) ---
    time = "10:00"
    location = "StudentCenter"

    prob = estimate_probability_majority_in_location_by_type(
        traces=traces,
        students_template=students,
        stype="athlete",
        location=location,
        time=time
    )

    print(f"P(majority of athletes in {location} at {time}) ≈ {prob:.3f}")

    # --- Print one sample trace in two views ---
    sample_trace = traces[0]
    print_trace_by_time(sample_trace, students)
    print_schedule_per_student(sample_trace, students)